# 실험 제목: 멀티턴 대화와 기억력 (Chat History)

**목표**: LLM API가 기본적으로 이전 대화를 기억하지 못함(Stateless)을 확인하고, 단순 리스트 조작 방식과 객체지향적인 `Memory` 클래스를 구현하여 문맥을 유지하는 방법을 실습합니다.

In [ ]:
# 필요한 라이브러리 import
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# .env 파일 로드 및 OpenAI 클라이언트 생성
load_dotenv()
client = OpenAI()

### 1. 기억력이 없는 단발성 호출 (Stateless)
API는 기본적으로 이전 요청을 기억하지 못합니다.

In [ ]:
# 첫 번째 질문
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕! 내 이름은 김철수야."}]
)
print("AI 답변 1:", response1.choices[0].message.content)

# 두 번째 질문 (단발성 호출로 이름 물어보기)
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "내 이름이 뭐라고 했지?"}]
)
print("AI 답변 2:", response2.choices[0].message.content)

### 2. 리스트(List)를 이용한 Chat History (가장 기본적인 방법)
이전 대화 내역을 리스트에 계속 `append`해서 보내는 기초적인 방식입니다.

In [ ]:
# 대화 기록을 저장할 리스트 생성
chat_history = [
    {"role": "system", "content": "당신은 사용자의 질문에 친절하게 답하는 챗봇입니다."}
]

user_msg_1 = "안녕! 내 이름은 김철수야."
chat_history.append({"role": "user", "content": user_msg_1})

res1 = client.chat.completions.create(model="gpt-4o-mini", messages=chat_history)
ai_msg_1 = res1.choices[0].message.content
print("AI 답변 1:", ai_msg_1)

chat_history.append({"role": "assistant", "content": ai_msg_1})

user_msg_2 = "내 이름이 뭐라고 했지?"
chat_history.append({"role": "user", "content": user_msg_2})

res2 = client.chat.completions.create(model="gpt-4o-mini", messages=chat_history)
ai_msg_2 = res2.choices[0].message.content
print("AI 답변 2:", ai_msg_2)

### 3. [심화] 객체 지향적인 SimpleMemory 클래스 구현
위의 리스트 `append` 방식은 직관적이지만 코드가 길어지면 관리가 어렵습니다.
향후 02_LangChain에서 배울 `ConversationBufferMemory`의 내부 동작 원리를 이해하기 위해, 순수 파이썬으로 간단한 Memory 객체를 만들어 봅니다.

In [ ]:
class SimpleMemory:
    def __init__(self, system_prompt="당신은 친절한 AI 어시스턴트입니다."):
        # 초기화 시 System Prompt를 세팅합니다.
        self.messages = [{"role": "system", "content": system_prompt}]
        
    def add_user_message(self, message):
        # 사용자의 메시지를 메모리에 추가합니다.
        self.messages.append({"role": "user", "content": message})
        
    def add_ai_message(self, message):
        # AI의 응답을 메모리에 추가합니다.
        self.messages.append({"role": "assistant", "content": message})
        
    def get_messages(self):
        # 현재까지 저장된 모든 메모리(대화 기록)를 반환합니다.
        return self.messages
        
    def clear(self):
        # 메모리를 초기화합니다. (System Prompt만 남김)
        system_prompt = self.messages[0]["content"]
        self.messages = [{"role": "system", "content": system_prompt}]

### 4. SimpleMemory를 활용한 대화 구현
클래스를 사용하면 코드가 얼마나 깔끔하고 직관적으로 변하는지 확인해 봅니다.

In [ ]:
# 메모리 객체 생성
memory = SimpleMemory(system_prompt="당신은 유머러스한 친구입니다. 반말로 짧게 답해주세요.")

# 1. 첫 번째 질문
memory.add_user_message("안녕! 난 홍길동이라고 해. 오늘 날씨가 진짜 덥다.")
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=memory.get_messages()
)
ai_reply1 = response1.choices[0].message.content
memory.add_ai_message(ai_reply1)  # 잊지 말고 AI 응답도 메모리에 추가!
print("AI 답변 1:", ai_reply1)

# 2. 두 번째 질문 (문맥 유지 확인)
memory.add_user_message("내 이름이 뭐라고?")
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=memory.get_messages()
)
ai_reply2 = response2.choices[0].message.content
memory.add_ai_message(ai_reply2)
print("AI 답변 2:", ai_reply2)

# 현재 메모리에 저장된 내용 확인
print("\n--- [현재 Memory 상태 확인] ---")
for msg in memory.get_messages():
    print(f"[{msg['role'].upper()}]: {msg['content']}")

### 실험 결과 정리

* **리스트 조작 vs 객체 지향**: 단순 리스트 `append` 방식은 기초를 이해하기 좋지만, `SimpleMemory` 같은 객체를 만들면 **데이터(메시지 리스트)와 조작(추가/조회/초기화)**을 한 곳(Class)에서 관리할 수 있어 코드가 깔끔해집니다.
* **프레임워크로의 연결**: 이 `SimpleMemory` 개념이 바로 LangChain에서 제공하는 `ConversationBufferMemory`나 `RunnableWithMessageHistory`의 동작 원리와 완벽하게 동일합니다.

### Notion 실험 로그 기록용

* **결과**: `list` 기반의 Chat History 관리를 객체 지향적인 `SimpleMemory` 클래스로 리팩토링하여 문맥을 성공적으로 유지함.
* **문제점**: 아무리 캡슐화를 잘 하더라도 결국 대화가 길어지면 Token 제한(Context Window)을 초과하게 됨.
* **다음 개선 방향**: LangChain의 `ConversationSummaryMemory`처럼, 일정 길이가 넘어가면 이전 내용을 LLM에게 요약(Summarize)시켜 메모리 크기를 줄이는 기법을 학습할 예정.